# 📓 Notebook 02 — Structured Output
---
**Phase 1 · LLM Fundamentals**

> Force LLMs to produce valid, reliable structured data — the bridge between AI and software engineering.

### What You'll Learn
| Topic | Why It Matters |
|-------|---------------|
| JSON mode | Guarantee valid JSON output |
| Function calling | Let LLMs invoke typed functions |
| Pydantic schemas | Runtime validation + type safety |
| Output guardrails | Catch hallucinations and malformed data |

### 🏗️ Build: LLM → Structured JSON Extractor (ETL-style)

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---
## 2. JSON Mode — Forcing Structured Output

### The Problem
LLMs generate free text by default. When you need structured data (for APIs, databases, pipelines), free text is unreliable.

### The Solution: `response_format`

```python
response_format={"type": "json_object"}   # Forces valid JSON output
```

> **Interview Tip:** *"How do you ensure an LLM always returns valid JSON?"*  
> Use `response_format={"type": "json_object"}` AND include "Respond in JSON" in your prompt (required by some models).

In [ ]:
# Without JSON mode — unreliable
result_raw = litellm.completion(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Extract the person's name, age, and city from: 'John Smith is 34 years old and lives in New York.'"}],
    temperature=0,
)
print("❌ Without JSON mode:")
print(result_raw.choices[0].message.content)
print()

# With JSON mode — guaranteed valid JSON
result_json = litellm.completion(
    model=DEFAULT_MODEL,
    messages=[{
        "role": "system", 
        "content": "You extract structured data. Always respond in valid JSON."
    }, {
        "role": "user", 
        "content": "Extract the person's name, age, and city from: 'John Smith is 34 years old and lives in New York.' Return JSON with keys: name, age, city"
    }],
    response_format={"type": "json_object"},
    temperature=0,
)
print("✅ With JSON mode:")
parsed = json.loads(result_json.choices[0].message.content)
print(json.dumps(parsed, indent=2))
print(f"Type: {type(parsed)}")

---
## 3. Function Calling / Tool Use

### What is Function Calling?
Function calling lets the LLM **decide which function to call** and **generate the arguments** — but YOU execute the function.

### Flow:
```
User Query → LLM decides tool + args → Your code executes tool → Result back to LLM → Final answer
```

> **Interview Critical:** The LLM does NOT execute functions. It only generates the function name and arguments as structured JSON. Your application code handles execution.

In [ ]:
# Define tools (functions the LLM can call)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name, e.g. 'London' or 'New York, NY'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a mathematical calculation",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression to evaluate, e.g. '(15 * 3) + 7'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("🔧 Defined 2 tools:")
for t in tools:
    print(f"  • {t['function']['name']}: {t['function']['description']}")

In [ ]:
# Let the LLM decide which tool to call
response = litellm.completion(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "What's the weather like in Tokyo?"}],
    tools=tools,
    tool_choice="auto",  # LLM decides whether to call a tool
    temperature=0,
)

# Check if LLM wants to call a tool
message = response.choices[0].message
if message.tool_calls:
    for tc in message.tool_calls:
        print(f"🔧 LLM wants to call: {tc.function.name}")
        print(f"   Arguments: {tc.function.arguments}")
        args = json.loads(tc.function.arguments)
        print(f"   Parsed: {args}")
else:
    print(f"💬 LLM responded directly: {message.content}")

In [ ]:
# Complete tool execution loop
def get_weather(location, unit="celsius"):
    """Simulated weather function."""
    # In production, this would call a real weather API
    weather_data = {
        "tokyo": {"temp": 22, "condition": "Partly Cloudy", "humidity": 65},
        "london": {"temp": 15, "condition": "Rainy", "humidity": 80},
        "new york": {"temp": 28, "condition": "Sunny", "humidity": 45},
    }
    data = weather_data.get(location.lower(), {"temp": 20, "condition": "Unknown", "humidity": 50})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32)
    return json.dumps({**data, "location": location, "unit": unit})

def calculate(expression):
    """Safe math evaluation."""
    try:
        # IMPORTANT: In production, use a proper math parser, not eval()
        allowed = set("0123456789+-*/().% ")
        if all(c in allowed for c in expression):
            return json.dumps({"expression": expression, "result": eval(expression)})
        return json.dumps({"error": "Invalid expression"})
    except Exception as e:
        return json.dumps({"error": str(e)})

# Map function names to actual functions
tool_functions = {
    "get_weather": get_weather,
    "calculate": calculate,
}

def run_with_tools(user_query):
    """Complete tool-calling loop."""
    messages = [{"role": "user", "content": user_query}]
    
    # Step 1: Ask LLM
    response = litellm.completion(
        model=DEFAULT_MODEL, messages=messages, tools=tools, tool_choice="auto", temperature=0
    )
    message = response.choices[0].message
    
    if not message.tool_calls:
        return message.content
    
    # Step 2: Execute tool calls
    messages.append(message)
    for tc in message.tool_calls:
        func = tool_functions[tc.function.name]
        args = json.loads(tc.function.arguments)
        result = func(**args)
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result,
        })
        print(f"  🔧 Called {tc.function.name}({args}) → {result}")
    
    # Step 3: Get final response with tool results
    final = litellm.completion(model=DEFAULT_MODEL, messages=messages, temperature=0)
    return final.choices[0].message.content

# Test it
print("📋 Tool Execution Demo")
print("=" * 60)
for query in ["What's the weather in London?", "Calculate (150 * 0.18) + 50"]:
    print(f"\n👤 {query}")
    answer = run_with_tools(query)
    print(f"🤖 {answer}")

---
## 4. Schema Enforcement with Pydantic

### Why Pydantic?
JSON mode guarantees valid JSON, but doesn't guarantee the **right structure**. Pydantic validates:
- Required fields exist
- Types are correct  
- Values are within constraints
- Custom business rules

> **Interview Tip:** *"How would you validate LLM output in production?"*  
> Use Pydantic models with `model_validate()` — you get type checking, required field validation, and custom validators. Retry with error feedback if validation fails.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum

class Sentiment(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative" 
    NEUTRAL = "neutral"

class ProductReview(BaseModel):
    """Schema for a parsed product review."""
    product_name: str = Field(description="Name of the product reviewed")
    rating: float = Field(ge=1.0, le=5.0, description="Rating from 1.0 to 5.0")
    sentiment: Sentiment = Field(description="Overall sentiment")
    pros: List[str] = Field(min_length=1, description="List of positive points")
    cons: List[str] = Field(default_factory=list, description="List of negative points")
    summary: str = Field(max_length=200, description="Brief summary under 200 chars")
    would_recommend: bool = Field(description="Whether reviewer would recommend")
    
    @field_validator("pros", "cons")
    @classmethod
    def non_empty_items(cls, v):
        return [item for item in v if item.strip()]

# Show the auto-generated JSON schema
print("📋 Pydantic JSON Schema:")
print(json.dumps(ProductReview.model_json_schema(), indent=2))

In [ ]:
def extract_with_schema(text, schema_class, max_retries=3):
    """Extract structured data from text using an LLM + Pydantic validation."""
    schema_json = json.dumps(schema_class.model_json_schema(), indent=2)
    
    prompt = f"""Extract structured information from the following text.
    
Respond with a JSON object matching this schema:
{schema_json}

Text: \"{text}\"
"""
    
    for attempt in range(max_retries):
        try:
            response = litellm.completion(
                model=DEFAULT_MODEL,
                messages=[
                    {"role": "system", "content": "You are a precise data extraction assistant. Always respond with valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0,
            )
            
            raw_json = json.loads(response.choices[0].message.content)
            validated = schema_class.model_validate(raw_json)
            print(f"  ✅ Validated on attempt {attempt + 1}")
            return validated
            
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                # Add error feedback to prompt for retry
                prompt += f"\n\nPrevious attempt failed with error: {e}. Please fix and try again."
    
    raise ValueError(f"Failed to extract valid data after {max_retries} attempts")

# Test extraction
review_text = """
I bought the Sony WH-1000XM5 headphones last month. The noise cancellation is absolutely 
incredible — best I've ever experienced. Sound quality is rich and balanced. Battery lasts 
about 30 hours which is fantastic. The only downsides are the price ($350 is steep) and 
they don't fold flat for travel like the XM4 did. Overall rating: 4.5 out of 5. Would 
definitely recommend for anyone who commutes or works in noisy environments.
"""

print("🔍 Extracting structured data from review...")
result = extract_with_schema(review_text, ProductReview)
print(f"\n📊 Parsed Review:")
print(f"  Product: {result.product_name}")
print(f"  Rating: {result.rating}/5")
print(f"  Sentiment: {result.sentiment.value}")
print(f"  Pros: {result.pros}")
print(f"  Cons: {result.cons}")
print(f"  Summary: {result.summary}")
print(f"  Recommend: {'✅ Yes' if result.would_recommend else '❌ No'}")

---
## 5. Output Guardrails

### Defense-in-Depth for LLM Output

| Layer | Check | Example |
|-------|-------|---------|
| 1. Format | Is it valid JSON? | `json.loads()` |
| 2. Schema | Does it match expected structure? | Pydantic validation |
| 3. Content | Are values reasonable? | Rating between 1-5, no profanity |
| 4. Business | Does it make business sense? | Price > 0, date in valid range |

> **Interview Tip:** *"How would you prevent hallucinated data in production?"*  
> Layer multiple guardrails: format validation → schema checking → content rules → confidence thresholds.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List

class CompanyInfo(BaseModel):
    """Validated company extraction schema."""
    name: str = Field(min_length=1, max_length=100)
    industry: str
    founded_year: Optional[int] = Field(default=None, ge=1800, le=2026)
    employees: Optional[int] = Field(default=None, ge=1)
    revenue_usd: Optional[float] = Field(default=None, ge=0)
    headquarters: Optional[str] = None
    key_products: List[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0, description="Extraction confidence 0-1")
    
    @field_validator("name")
    @classmethod
    def name_not_generic(cls, v):
        if v.lower() in ["unknown", "n/a", "none", "company"]:
            raise ValueError("Company name cannot be generic placeholder")
        return v

def extract_with_guardrails(text, schema_class):
    """Extract with full guardrail chain."""
    # Layer 1: Get LLM output
    response = litellm.completion(
        model=DEFAULT_MODEL,
        messages=[
            {"role": "system", "content": "Extract company information. Include a confidence score (0-1) indicating how certain you are. Respond in JSON."},
            {"role": "user", "content": f"Extract company info from: {text}"}
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    raw = response.choices[0].message.content
    
    # Layer 2: JSON parse
    try:
        data = json.loads(raw)
        print("  ✅ Layer 1: Valid JSON")
    except json.JSONDecodeError as e:
        print(f"  ❌ Layer 1: Invalid JSON — {e}")
        return None
    
    # Layer 3: Schema validation
    try:
        validated = schema_class.model_validate(data)
        print("  ✅ Layer 2: Schema valid")
    except Exception as e:
        print(f"  ❌ Layer 2: Schema error — {e}")
        return None
    
    # Layer 4: Confidence check
    if validated.confidence < 0.5:
        print(f"  ⚠️ Layer 3: Low confidence ({validated.confidence}) — flagged for human review")
    else:
        print(f"  ✅ Layer 3: Confidence OK ({validated.confidence})")
    
    return validated

# Test
text = "Apple Inc., founded in 1976 by Steve Jobs, is a technology company headquartered in Cupertino, CA. They employ about 164,000 people and had $383 billion in revenue in 2023. Key products include iPhone, Mac, iPad, and Apple Watch."

print("🛡️ Guardrailed Extraction")
print("=" * 50)
result = extract_with_guardrails(text, CompanyInfo)
if result:
    print(f"\n📊 Result: {result.model_dump_json(indent=2)}")

---
## 📝 Interview Quiz — Structured Output

### Q1: What is the difference between "function calling" and "tool use"?

<details>
<summary>💡 Answer</summary>

They are essentially the same concept with different naming:
- **OpenAI** calls it "function calling" (legacy) → "tool use" (current)
- **Anthropic** calls it "tool use"
- **Google** calls it "function calling"

The LLM generates a function name + arguments as JSON. Your code executes the function. The result is fed back to the LLM for a final response.

Key: The **LLM never executes code**. It only generates structured intent.
</details>

### Q2: Why use Pydantic instead of just `json.loads()`?

<details>
<summary>💡 Answer</summary>

`json.loads()` only validates JSON syntax. Pydantic validates:
- **Required fields** exist
- **Types** are correct (string, int, float)
- **Constraints** are met (min/max values, string length)
- **Enums** are valid values
- **Custom rules** via validators
- **Nested structures** are fully validated

Production systems need all of these to prevent invalid data from entering your pipeline.
</details>

### Q3: How would you handle LLM extraction failures in production?

<details>
<summary>💡 Answer</summary>

Multi-layer retry strategy:
1. **Retry with feedback** — Send the validation error back to the LLM and ask it to fix
2. **Fallback model** — Try a different/larger model
3. **Graceful degradation** — Return partial data with confidence flags
4. **Human-in-the-loop** — Low-confidence extractions go to human review queue
5. **Circuit breaker** — After N failures, stop and alert instead of burning API credits
</details>

### Q4: What is `tool_choice` and what are its options?

<details>
<summary>💡 Answer</summary>

`tool_choice` controls whether/how the LLM uses tools:
- `"auto"` — LLM decides whether to call a tool or respond directly
- `"none"` — Never call tools, always respond in text
- `"required"` — Must call at least one tool
- `{"type": "function", "function": {"name": "..."}}` — Force a specific tool

**When to use each:**
- `auto`: General agents, the LLM routes naturally
- `required`: When the task always needs a tool (e.g., DB lookup)
- Specific: When you know exactly which tool is needed
</details>

### Q5: Explain the complete flow of a function-calling interaction.

<details>
<summary>💡 Answer</summary>

```
1. User sends query → e.g., "What's the weather in Paris?"
2. LLM receives query + tool definitions
3. LLM outputs: tool_calls = [{name: "get_weather", args: {location: "Paris"}}]
4. Your code extracts the function name and arguments
5. Your code executes: get_weather(location="Paris") → {"temp": 18, ...}
6. Tool result is sent back to LLM as a "tool" message
7. LLM generates final natural-language response using the tool result
8. Response sent to user: "It's currently 18°C in Paris."
```

The LLM acts as a router/planner; your code is the executor.
</details>

### Q6: How would you build a robust ETL pipeline using LLM extraction?

<details>
<summary>💡 Answer</summary>

```
Source Documents → Chunking → LLM Extraction → Schema Validation → Dedup → Load
```

Key considerations:
- **Pydantic schemas** per document type
- **Retry logic** with error feedback (3 attempts typical)
- **Confidence scores** from the LLM for each extraction
- **Human review queue** for low-confidence items
- **Logging** every extraction attempt for debugging
- **Batch processing** to manage costs
- **Idempotency** — same input always produces same output (temp=0)
</details>

---
## 🏗️ BUILD: LLM → Structured JSON Extractor (ETL-style)

A production-grade extraction pipeline that takes unstructured text and outputs validated, typed data objects.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum
import time

class JobLevel(str, Enum):
    JUNIOR = "junior"
    MID = "mid"
    SENIOR = "senior"
    LEAD = "lead"
    MANAGER = "manager"
    DIRECTOR = "director"
    VP = "vp"
    C_LEVEL = "c_level"

class JobPosting(BaseModel):
    title: str = Field(min_length=2, max_length=200)
    company: str = Field(min_length=1)
    location: str
    remote: bool = Field(description="Is this a remote position?")
    salary_min: Optional[int] = Field(default=None, ge=0)
    salary_max: Optional[int] = Field(default=None, ge=0)
    currency: str = Field(default="USD", max_length=3)
    level: JobLevel
    skills: List[str] = Field(min_length=1)
    experience_years: Optional[int] = Field(default=None, ge=0, le=50)
    benefits: List[str] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)
    
    @field_validator("salary_max")
    @classmethod
    def max_gte_min(cls, v, info):
        if v is not None and info.data.get("salary_min") is not None:
            if v < info.data["salary_min"]:
                raise ValueError("salary_max must be >= salary_min")
        return v

class ExtractionResult:
    def __init__(self, data, raw_response, attempts, latency_ms):
        self.data = data
        self.raw = raw_response
        self.attempts = attempts
        self.latency_ms = latency_ms
        self.status = "success" if data else "failed"

def extract_pipeline(text, schema_class, model=None, max_retries=3):
    """Production ETL extraction pipeline."""
    model = model or DEFAULT_MODEL
    schema_json = json.dumps(schema_class.model_json_schema(), indent=2)
    
    system = f"""You are a precise data extraction engine. 
Extract structured data from the given text.
Include a 'confidence' score (0.0-1.0) indicating extraction certainty.
Respond ONLY with valid JSON matching this schema:
{schema_json}"""
    
    prompt = f"Extract structured data from this text:\n\n{text}"
    start = time.time()
    
    for attempt in range(1, max_retries + 1):
        try:
            response = litellm.completion(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0,
            )
            raw = response.choices[0].message.content
            data = json.loads(raw)
            validated = schema_class.model_validate(data)
            latency = int((time.time() - start) * 1000)
            return ExtractionResult(validated, raw, attempt, latency)
            
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt}: {e}")
            prompt += f"\n\nPrevious attempt error: {e}. Fix and retry."
    
    latency = int((time.time() - start) * 1000)
    return ExtractionResult(None, None, max_retries, latency)

# Test with real-world job postings
job_texts = [
    """Senior Machine Learning Engineer at Stripe (Remote, US). $180,000-$250,000/year.
    We're looking for an experienced ML engineer to join our fraud detection team. 
    5+ years experience with Python, PyTorch, and production ML systems required.
    Strong background in anomaly detection preferred. Benefits include unlimited PTO, 
    401k match, health/dental/vision, and $10K learning budget.""",
    
    """Junior Frontend Developer - TechStartup Inc, London UK. £35,000-£45,000.
    React, TypeScript, and CSS skills needed. 1-2 years experience. 
    Hybrid role (3 days office). Benefits: free lunch, gym membership.""",
]

print("🏭 ETL Extraction Pipeline")
print("=" * 60)
for i, text in enumerate(job_texts, 1):
    print(f"\n--- Job {i} ---")
    result = extract_pipeline(text, JobPosting)
    if result.data:
        d = result.data
        print(f"  ✅ {d.title} @ {d.company}")
        print(f"     Level: {d.level.value} | Remote: {d.remote}")
        print(f"     Salary: {d.salary_min}-{d.salary_max} {d.currency}")
        print(f"     Skills: {', '.join(d.skills)}")
        print(f"     Confidence: {d.confidence} | Attempts: {result.attempts} | Latency: {result.latency_ms}ms")
    else:
        print(f"  ❌ Extraction failed after {result.attempts} attempts")

---
## ✅ Notebook 02 Summary

| Concept | Key Takeaway |
|---------|-------------|
| JSON mode | `response_format={"type": "json_object"}` guarantees valid JSON |
| Function calling | LLM generates function name + args, your code executes |
| `tool_choice` | `auto`, `none`, `required`, or force specific tool |
| Pydantic | Runtime validation with types, constraints, and custom rules |
| Guardrails | Layer: format → schema → content → business rules |
| Retry strategy | Send error feedback back to LLM for self-correction |
| ETL pattern | Chunk → Extract → Validate → Retry → Load |

### ➡️ Next: [Notebook 03 — Embeddings](./03_embeddings.ipynb)
*Learn vector representations, similarity search, and build a semantic search engine.*